# 00 Preparación del equipo 8

Ejecutar una vez con el propietario del esquema y del volumen. Reutiliza el WPI, Bronze y su checkpoint. Crea silver, cuarentena, las métricas y las tablas de oro en `default`. El job diario empieza en 01.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))


In [ ]:
%run ./_config


## Volume y carpetas

`raw_zips/` guarda el zip, `landing/` el CSV que lee bronze, `checkpoints/` los
checkpoints de los streams, `ref/` el World Port Index y `_schemas/` el esquema de Auto Loader.


In [ ]:
import os

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
COMMENT 'Archivos de OceanWatch: zips NOAA, CSV de landing, checkpoints y referencias.'
""")

for ruta in (RAW_ZIPS, LANDING, CHECKPOINTS, SCHEMAS, REF):
    os.makedirs(ruta, exist_ok=True)
    print("Listo", ruta)


## Tablas

Bronze conserva los mensajes y su procedencia. Silver agrupa físicamente por fecha, guarda `batch_alta` y conserva una observación válida por MMSI e instante. Las copias inválidas llegan a cuarentena con sus causas. `oceanwatch_metricas` recibe el resumen de Bronze y, con `etapa = 'silver'`, el conteo de cada batch de silver. En `default`, 04 publica trayectorias, presencia portuaria y tráfico. `05_ml` publica las evaluaciones de velocidad. La presencia no guarda el número WPI: si la tabla ya tiene la columna `numero_puerto`, bórrala y vuelve a ejecutar esta celda antes de `04`.


In [ ]:
tablas = {
    BRONZE: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING COMMENT 'Ruta del CSV en landing.',
          _ingested_at TIMESTAMP COMMENT 'Instante en que el batch escribio la fila.'
        )
        USING DELTA
        COMMENT 'Bronze AIS: mensaje crudo, tal como llego. Cada archivo se lee una vez. La deduplicacion es en silver.'
    """,
    SILVER: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          fecha_dia DATE,
          batch_alta BIGINT
        )
        USING DELTA
        PARTITIONED BY (fecha_dia)
        COMMENT 'Silver AIS: una fila por (MMSI, BaseDateTime) en cada fecha_dia. La deduplicacion es aqui.'
    """,
    CUARENTENA: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          motivo STRING COMMENT 'Reglas fallidas, separadas por coma.'
        )
        USING DELTA
        COMMENT 'Filas AIS que no entran a silver, con el motivo.'
    """,
    METRICAS: """
        (
          fecha DATE COMMENT 'Dia del parametro del job.',
          etapa STRING COMMENT 'bronze o silver.',
          filas_antes BIGINT COMMENT 'Conteo de la tabla bronze antes del stream.',
          filas_despues BIGINT COMMENT 'Conteo de la tabla bronze despues del stream.',
          filas_entrada BIGINT COMMENT 'Filas del batch de silver.',
          filas_validas BIGINT COMMENT 'Filas del batch que pasan las reglas.',
          filas_cuarentena BIGINT COMMENT 'Filas del batch que van a cuarentena.',
          ejecutado_en TIMESTAMP
        )
        USING DELTA
        COMMENT 'Conteos de cada ejecucion de bronze y de cada batch de silver.'
    """,
    PUERTOS: """
        (
          numero INT COMMENT 'World Port Index Number.',
          nombre STRING,
          pais STRING,
          latitud DOUBLE,
          longitud DOUBLE
        )
        USING DELTA
        COMMENT 'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'
    """,
    GOLD_TRAY: """
        (
          fecha DATE,
          MMSI STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP,
          lat_inicio DOUBLE,
          lon_inicio DOUBLE,
          lat_fin DOUBLE,
          lon_fin DOUBLE,
          distancia_km DOUBLE COMMENT 'Suma de tramos haversine del dia, radio 6371 km.',
          sog_media DOUBLE
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Una trayectoria por buque y dia, calculada sobre silver.'
    """,
    GOLD_PUERTO: """
        (
          fecha DATE,
          MMSI STRING,
          puerto STRING,
          pais STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Buque con al menos una posicion del dia junto a un puerto del WPI.'
    """,
    GOLD_TRAFICO: """
        (
          fecha DATE,
          hora INT COMMENT 'Hora UTC de BaseDateTime.',
          h3_cell STRING COMMENT 'Celda H3 resolucion 8, la misma de la Parte 1.',
          n_posiciones BIGINT,
          n_buques BIGINT
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Matriz de trafico celda H3 por hora, un dia.'
    """,
    CONTROL_ORO: """(fecha DATE, batch_plata BIGINT, publicado_en TIMESTAMP)
        USING DELTA COMMENT 'Último batch de Silver publicado en los tres productos diarios.'""",
    GOLD_ANOM: """(fecha DATE, MMSI STRING, n_posiciones BIGINT, sog_media DOUBLE,
        velocidad_tramos DOUBLE, brecha DOUBLE, score DOUBLE, es_anomalia BOOLEAN, estado STRING,
        modelo_version STRING, run_id STRING) USING DELTA PARTITIONED BY (fecha)
        COMMENT 'Evaluación de velocidad por buque y día, cobertura y versión MLflow.'""",
    CONTROL_ML: """(fecha DATE, revision_oro STRING, publicado_en TIMESTAMP) USING DELTA
        COMMENT 'Versiones de los días de oro usados en entrenamiento e inferencia.'"""
}

for nombre, cuerpo in tablas.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {nombre} {cuerpo}")
    print("Tabla", nombre)
spark.sql(f"ALTER TABLE {BRONZE} SET TBLPROPERTIES ('delta.appendOnly'='true')")


## World Port Index

La Parte 1 usó `UpdatedPub150.csv` (`Main Port Name`, `Country Code`, `Latitude`,
`Longitude`). El número de puerto en el CSV viene como `7950.0`: se lee como
double y se guarda como int. Spark no puede leer el archivo del repo (eso pide
ANY FILE), así que Python lo copia desde la carpeta Git a `ref/` y la lectura
es sobre `/Volumes`.


In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import functions as F

destino = Path(REF) / "UpdatedPub150.csv"
destino.parent.mkdir(parents=True, exist_ok=True)

def csv_en_repo() -> Path | None:
    """CSV en la raiz del repo, al lado de src/, visto desde la carpeta Git."""
    try:
        ruta_nb = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
    except Exception:
        return None
    bases = [ruta_nb]
    if not ruta_nb.startswith("/Workspace"):
        bases.append("/Workspace" + ruta_nb)
    # src/00_setup -> raiz del repo. Se prueban las dos formas del path.
    for base in bases:
        candidato = Path(base).parent.parent / "UpdatedPub150.csv"
        if candidato.is_file():
            return candidato
    return None

origen = csv_en_repo()
if origen is not None:
    shutil.copyfile(origen, destino)
    print("Copiado", origen, "->", destino)
elif not destino.is_file():
    raise FileNotFoundError(
        "No esta UpdatedPub150.csv en el repo ni en "
        f"{destino}. Abre el notebook desde la carpeta Git del repo."
    )
else:
    print("Se usa la copia ya presente en", destino)

# El numero viene como '7950.0'. String -> int falla. Double y luego int.
(
    spark.read.option("header", True).csv(str(destino))
    .select(
        F.expr("try_cast(`World Port Index Number` AS DOUBLE)").cast("int").alias("numero"),
        F.col("Main Port Name").alias("nombre"),
        F.col("Country Code").alias("pais"),
        F.expr("try_cast(Latitude AS DOUBLE)").alias("latitud"),
        F.expr("try_cast(Longitude AS DOUBLE)").alias("longitud"),
    )
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull() & F.col("numero").isNotNull())
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PUERTOS)
)
spark.sql(
    f"COMMENT ON TABLE {PUERTOS} IS "
    "'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'"
)
print("Puertos", spark.table(PUERTOS).count())


## Permisos

`mine-group8`, `h.benitez@uniandes.edu.co` y `l.galvan@uniandes.edu.co` reciben los mismos privilegios sobre objetos de este proyecto: lectura y escritura del volume, `SELECT` y `MODIFY` en las tablas. `EXECUTE` del modelo se concede aquí si ya está registrado, y de nuevo en `05` al registrar la primera versión. El catálogo es de la universidad: no se otorga nada a nivel de catálogo ni de esquema.


In [ ]:
principales = (
    "`mine-group8`",
    "`h.benitez@uniandes.edu.co`",
    "`l.galvan@uniandes.edu.co`",
)

for principal in principales:
    spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {principal}")
    spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {principal}")
    print("GRANT volume", principal)

for tabla in tablas:
    for principal in principales:
        spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {principal}")
        spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {principal}")
    print("GRANT", tabla)

import mlflow
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

mlflow.set_registry_uri("databricks-uc")
try:
    MlflowClient().get_registered_model(MODELO)
except MlflowException:
    print("El modelo aun no esta registrado. 05_ml concede EXECUTE cuando registra la primera version.")
else:
    citado = ".".join(f"`{parte}`" for parte in MODELO.replace("`", "").split("."))
    for principal in principales:
        try:
            spark.sql(f"GRANT EXECUTE ON FUNCTION {citado} TO {principal}")
            print("GRANT EXECUTE", citado, principal)
        except Exception as exc:
            print("No se pudo conceder EXECUTE", principal, exc)


## Comentarios de oro

Actualiza los comentarios y las propiedades de las cuatro tablas de oro. Si `grupo_lectura` tiene un grupo distinto de `mine-group8`, le concede `SELECT` sobre esas tablas. El catálogo y el esquema no se tocan.


In [ ]:
OROS = [GOLD_TRAY, GOLD_PUERTO, GOLD_TRAFICO, GOLD_ANOM]

def gobierno(lector):
    if lector and lector.strip().lower() == "mine-group8":
        raise ValueError("Indica un grupo analítico distinto del grupo de ingeniería.")
    if lector:
        # La comilla invertida se escapa como identificador de Unity Catalog.
        rol = "`" + lector.replace("`", "``") + "`"
        # El catálogo es de la universidad. Solo se concede SELECT en las tablas de oro.
        for t in OROS:
            spark.sql(f"GRANT SELECT ON TABLE {t} TO {rol}")
    else:
        print("El widget grupo_lectura está vacío. Se inspeccionan los permisos existentes.")
    dic = {
        "fecha":"Día UTC del evento AIS.", "MMSI":"Identificador AIS del buque, nueve dígitos.",
        "n_posiciones":"Mensajes válidos dentro de la unidad de esta tabla.", "inicio":"Primera señal UTC.",
        "fin":"Última señal UTC.", "lat_inicio":"Latitud inicial, grados WGS84.", "lon_inicio":"Longitud inicial, grados WGS84.",
        "lat_fin":"Latitud final, grados WGS84.", "lon_fin":"Longitud final, grados WGS84.",
        "distancia_km":"Suma de tramos haversine diarios en kilómetros.", "sog_media":"Media de SOG por mensaje, en nudos.",
        "puerto":"Nombre del puerto en WPI.", "pais":"País según WPI.",
        "hora":"Hora UTC entre 0 y 23.", "h3_cell":"Identificador hexadecimal H3 de resolución 8.",
        "n_buques":"MMSI distintos en la celda y hora.", "velocidad_tramos":"Distancia observada dividida por tiempo total, en nudos.",
        "brecha":"Diferencia absoluta de velocidades, en nudos.", "score":"Opuesto de decision_function; valor positivo indica alerta.",
        "es_anomalia":"Alerta del modelo. Nulo si el buque no fue evaluado.", "estado":"evaluado, sin_historial o traza_insuficiente.",
        "modelo_version":"Versión registrada utilizada para puntuar la jornada.", "run_id":"Run MLflow de entrenamiento."
    }
    for t in OROS:
        for columna in spark.table(t).columns:
            spark.sql(f"ALTER TABLE {t} ALTER COLUMN `{columna}` COMMENT '{dic[columna]}'")
        spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('oceanwatch.contrato'='2.0', "
                  "'oceanwatch.frecuencia'='diaria', 'oceanwatch.huso'='UTC', "
                  "'oceanwatch.responsable'='mine-group8')")
        display(spark.sql(f"SHOW GRANTS ON TABLE {t}"))
    display(spark.sql(f"SHOW GRANTS ON TABLE {SILVER}"))

dbutils.widgets.text("grupo_lectura", "", "Grupo lector opcional, distinto de mine-group8")
gobierno(dbutils.widgets.get("grupo_lectura").strip())


## Reiniciar todo (opcional)

Esta celda no hace nada mientras `REINICIAR_TODO` sea `False`.

Con `True` borra las tablas de este setup, el modelo `oceanwatch_detector_velocidad` si existe, el contenido de `checkpoints/` y los archivos que el pipeline escribe en `landing/` y `evidencias/`. Los zip de `raw_zips/` se quedan, así `01_ingesta` no vuelve a bajar NOAA.

Pasos:

1. Ejecuta las celdas de configuración.
2. Pon `REINICIAR_TODO = True` y ejecuta solo esta celda.
3. Vuelve a dejar `REINICIAR_TODO = False`.
4. Ejecuta otra vez volume, tablas, World Port Index y permisos.
5. Ejecuta `01`, `02`, `03`, `04` y `05`, en ese orden.


In [ ]:
import shutil
from pathlib import Path
import mlflow
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

REINICIAR_TODO = False

TABLAS = (
    BRONZE,
    SILVER,
    CUARENTENA,
    METRICAS,
    PUERTOS,
    GOLD_TRAY,
    GOLD_PUERTO,
    GOLD_TRAFICO,
    GOLD_ANOM,
    CONTROL_ORO,
    CONTROL_ML,
)


def vaciar_directorio(ruta: Path) -> None:
    if not ruta.exists():
        print("No existe", ruta)
        return
    for hijo in ruta.iterdir():
        if hijo.is_dir():
            shutil.rmtree(hijo)
        else:
            hijo.unlink()
        print("Borrado", hijo)


if not REINICIAR_TODO:
    print(
        "Opcional. Pon REINICIAR_TODO = True para borrar las tablas oceanwatch_, "
        "el modelo, checkpoints, landing y evidencias. Los zip de raw_zips se conservan."
    )
else:
    for tabla in TABLAS:
        spark.sql(f"DROP TABLE IF EXISTS {tabla}")
        print("DROP", tabla)
    # Unity Catalog no acepta DROP MODEL en spark.sql. El borrado va por el registro.
    mlflow.set_registry_uri("databricks-uc")
    try:
        MlflowClient().delete_registered_model(MODELO)
        print("DROP MODEL", MODELO)
    except MlflowException as error:
        if getattr(error, "error_code", "") != "RESOURCE_DOES_NOT_EXIST" and "does not exist" not in str(error).lower():
            raise
        print("El modelo no existe", MODELO)
    vaciar_directorio(Path(CHECKPOINTS))
    vaciar_directorio(Path(LANDING))
    vaciar_directorio(Path(VOLUME_ROOT) / "evidencias")
    print("raw_zips se conserva:", RAW_ZIPS)
    print("Listo. Ejecuta volume, tablas, WPI y permisos. Despues 01, 02, 03, 04 y 05.")
